In [9]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import xarray
sys.path.append("../src/climate_trends/")

import load as ld
import xarray as xr
import pandas as pd
from statsmodels.tsa.seasonal import STL
import emcee
from statsmodels.tsa.statespace.structural import UnobservedComponents


# Summary

In this notebook, We go further than the Bayesian parametric fitting and fit for a State space model.

# Load the data [for Berlin and Paris]

In [61]:
ds_p=ld.load_single_file("../data/raw/era5_paris_t2m.nc")
t2m_p = ds_p["t2m"].sel(latitude=48.75,longitude=2.25,method='nearest') -273.15
time_dt = pd.to_datetime(t2m_p["valid_time"].values)
time = (time_dt.year + (time_dt.month - 1)/12.0)
paris_panda = t2m_p.to_series().dropna()
##### Normalize time axis ######
### Normalize time axis ##############
time_mean = np.mean(time)
time_std  = np.std(time)
time_norm = (time - time_mean) / time_std


# State Space Model

In [66]:
time_std

np.float64(21.650621730017274)

## STL decomposition


In [ ]:
stl = STL(t2m_p, period=12, robust=True)  
stl_result = stl.fit()
temp_trend=stl_result.trend
temp_seasonal=stl_result.seasonal
###########
temp_deseason = (t2m_p - temp_seasonal).values

# State Space model

## CaseI: State Spcae Model - Local linear trend
I fit a local linear trend model with AR(1) residuals on the deseasoned t2m data. Theis model identifies strong temporal persistence (r$\phi$=0.95). While the model allows for a time-varying warming rate, the trend variance is poorly constrained and not significantly different from zero, hence we have limited evidence for warming trend over time.

In [54]:
model = UnobservedComponents(temp_deseason,level='local linear trend', autoregressive=1)

In [55]:
print(result.model.state_names)
print(result.summary())
state_names = result.model.state_names
trend_idx = state_names.index('trend')

slope = result.smoothed_state[trend_idx]

['level', 'trend', 'ar.L1']
                        Unobserved Components Results                         
Dep. Variable:                      y   No. Observations:                  900
Model:             local linear trend   Log Likelihood               -2406.129
                              + AR(1)   AIC                           4822.257
Date:                Mon, 01 Jun 2026   BIC                           4846.258
Time:                        08:27:29   HQIC                          4831.427
Sample:                             0                                         
                                - 900                                         
Covariance Type:                  opg                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
sigma2.irregular     1.3956      0.482      2.895      0.004       0.451       2.341
sigma2

# CaseII: SARIMA 
For a second case, I fit the data fit a SARIMA(1,0,0)model. What I find is: that the model estimates a warming rate of $0.26 \pm 0.06^\circ\mathrm{C}$ per decade. However, the uncertainty is large, and the trend coefficient is not statistically distinguishable from zero (p = 0.868). It is possible that this weak constrain is due to the correlated to the AR(1) modeling.

In [67]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    temp_deseason,
    exog=time_norm,
    order=(1,0,0)
)

result = model.fit()

In [68]:
print(result.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                  900
Model:               SARIMAX(1, 0, 0)   Log Likelihood               -1841.417
Date:                Mon, 01 Jun 2026   AIC                           3688.833
Time:                        09:32:58   BIC                           3703.240
Sample:                             0   HQIC                          3694.337
                                - 900                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
x1             0.5817      3.510      0.166      0.868      -6.298       7.461
ar.L1          0.9857      0.006    164.651      0.000       0.974       0.997
sigma2         3.4911      0.117     29.770      0.0

## Warming trend

In [ ]:
trend=result.params[0]
warming_trend_per_year=trend/time_std
print("The warming trend per decade is:",warming_trend_per_year*10)

The warming trend per decade is: 0.2686661157367492
